# Quantization with Optimum Quanto - Step-by-Step Guide

This guide demonstrates how to quantize neural networks using Optimum Quanto to reduce model size and memory usage while maintaining performance.

## Overview
Quantization is a technique that reduces the precision of model weights from 32-bit floating point to lower precision formats (like 8-bit integers), significantly reducing model size and memory requirements.

## Step 1: Environment Setup and Installation

In [3]:
# Install the required quantization library
!pip install optimum-quanto

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 15.2 MB/s eta 0:00:00


**What it does:** Installs Optimum Quanto, a library for quantizing transformer models efficiently.

## Step 2: Import Required Libraries

In [4]:
import re
import torch
from collections import defaultdict

from optimum.quanto import quantize, qint8, freeze
from transformers import AutoModelForCausalLM, AutoTokenizer

**Libraries explained:**
- `optimum.quanto`: Main quantization library
- `transformers`: Hugging Face transformers for model loading
- `torch`: PyTorch for tensor operations
- `collections.defaultdict`: For efficient dictionary operations
- `re`: Regular expressions for string parsing

## Step 3: Authentication (Optional)

**Note:** This step is only required if you want to use gated models like Llama. For open models like Pythia, you can skip this step.

In [5]:
# Optional: Only needed for gated models like Llama
!huggingface-cli login


Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



## Step 4: Model Selection and Loading

In [6]:
# Choose your model based on access
model_name = "meta-llama/Llama-3.2-1B-Instruct"  # If you have access to Llama
# model_name = "EleutherAI/pythia-410m"           # Alternative open model

# Load the model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

**Parameters:**
- `model_name`: String identifier for the model on Hugging Face Hub
- `device_map="cuda"`: Automatically maps model to GPU memory

**What it does:** Loads the pre-trained model and its corresponding tokenizer into GPU memory.

## Step 5: Inspect Original Model Structure

In [7]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [8]:
if model_name == "meta-llama/Llama-3.2-1B-Instruct":
    print(model.model.layers[0].self_attn.q_proj.weight)
else:
    print(model.gpt_neox.layers[0].attention.dense.weight)

Parameter containing:
tensor([[-0.0179,  0.0066,  0.0247,  ..., -0.0087, -0.0117,  0.0201],
        [ 0.0122,  0.0593,  0.0552,  ..., -0.0332, -0.0154,  0.0108],
        [ 0.0178,  0.0155,  0.0344,  ..., -0.0386, -0.0386, -0.0276],
        ...,
        [ 0.0298,  0.0352,  0.0713,  ..., -0.0718, -0.0265, -0.0287],
        [ 0.0226, -0.0248,  0.0352,  ..., -0.0120, -0.0287, -0.0148],
        [-0.0258, -0.0537, -0.0131,  ...,  0.0542,  0.0096, -0.0028]],
       device='cuda:0', dtype=torch.bfloat16, requires_grad=True)


**What it does:**
- Displays the model architecture
- Shows the weight tensor of the first attention layer's query projection or dense if you're using pythia-410m model
- Helps understand the original model structure before quantization

## Step 6: Utility Functions for Model Analysis

### Function 1: Enhanced Tensor Enumeration

In [9]:
def named_module_tensors(module, recurse=False):
    """
    Enhanced version of named_parameters that handles quantized tensors.

    Args:
        module (torch.nn.Module): The module to inspect
        recurse (bool): Whether to recursively check submodules

    Yields:
        tuple: (name, tensor) pairs for all tensors in the module

    Note: This function is monkey-patched to work with Quanto's quantized tensors
    that have _data and _scale attributes.
    """
    for named_parameter in module.named_parameters(recurse=recurse):
        name, val = named_parameter
        flag = True
        if hasattr(val,"_data") or hasattr(val,"_scale"):
            if hasattr(val,"_data"):
                yield name + "._data", val._data
            if hasattr(val,"_scale"):
                yield name + "._scale", val._scale
        else:
            yield named_parameter

    for named_buffer in module.named_buffers(recurse=recurse):
        yield named_buffer

### Function 2: Data Type Size Calculator

In [10]:
def dtype_byte_size(dtype):
    """
    Calculate the size in bytes for a given PyTorch data type.

    Args:
        dtype (torch.dtype): PyTorch data type (e.g., torch.float32, torch.int8)

    Returns:
        float: Size in bytes per element

    Raises:
        ValueError: If dtype is not valid

    Examples:
        >>> dtype_byte_size(torch.float32)
        4.0
        >>> dtype_byte_size(torch.int8)
        1.0
    """
    if dtype == torch.bool:
        return 1 / 8
    bit_search = re.search(r"[^\d](\d+)$", str(dtype))
    if bit_search is None:
        raise ValueError(f"`dtype` is not a valid dtype: {dtype}.")
    bit_size = int(bit_search.groups()[0])
    return bit_size // 8

### Function 3: Module Size Computation

In [11]:
def compute_module_sizes(model):
    """
    Compute the memory footprint of each module in the model.

    Args:
        model (torch.nn.Module): The model to analyze

    Returns:
        defaultdict: Dictionary with module names as keys and sizes in bytes as values

    Note: Includes both the full model size ('') and individual component sizes
    """
    module_sizes = defaultdict(int)
    for name, tensor in named_module_tensors(model, recurse=True):
        size = tensor.numel() * dtype_byte_size(tensor.dtype)
        name_parts = name.split(".")
        for idx in range(len(name_parts) + 1):
            module_sizes[".".join(name_parts[:idx])] += size

    return module_sizes

## Step 7: Analyze Original Model Size

In [13]:
module_sizes = compute_module_sizes(model)
print(module_sizes)
print(f"The model size is {module_sizes[''] * 1e-9} GB")

defaultdict(<class 'int'>, {'': 2471629056, 'model': 2471629056, 'model.embed_tokens': 525336576, 'model.embed_tokens.weight': 525336576, 'model.layers': 1946288128, 'model.layers.0': 121643008, 'model.layers.0.self_attn': 20971520, 'model.layers.0.self_attn.q_proj': 8388608, 'model.layers.0.self_attn.q_proj.weight': 8388608, 'model.layers.0.self_attn.k_proj': 2097152, 'model.layers.0.self_attn.k_proj.weight': 2097152, 'model.layers.0.self_attn.v_proj': 2097152, 'model.layers.0.self_attn.v_proj.weight': 2097152, 'model.layers.0.self_attn.o_proj': 8388608, 'model.layers.0.self_attn.o_proj.weight': 8388608, 'model.layers.0.mlp': 100663296, 'model.layers.0.mlp.gate_proj': 33554432, 'model.layers.0.mlp.gate_proj.weight': 33554432, 'model.layers.0.mlp.up_proj': 33554432, 'model.layers.0.mlp.up_proj.weight': 33554432, 'model.layers.0.mlp.down_proj': 33554432, 'model.layers.0.mlp.down_proj.weight': 33554432, 'model.layers.0.input_layernorm': 4096, 'model.layers.0.input_layernorm.weight': 4096

**What it does:**
- Calculates the memory footprint of each module
- Displays total model size in GB
- Provides baseline measurements before quantization

## Step 8: Test Original Model Performance

In [14]:
print("Before generate:")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e6:.2f} MB")

text = "What is Machine Learning?"
inputs = tokenizer(text, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

print("After generate:")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e6:.2f} MB")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Before generate:
Allocated: 2472.68 MB
What is Machine Learning? and What are its Applications?
Machine learning is a subfield of artificial intelligence (AI) that involves training algorithms to make predictions or decisions based on data. The data is used to identify patterns, relationships, and trends that can be used to make predictions
After generate:
Allocated: 2481.20 MB


**Parameters:**
- `max_new_tokens=50`: Maximum number of new tokens to generate
- `skip_special_tokens=True`: Excludes special tokens from decoded output

**What it does:**
- Monitors GPU memory usage before and after generation
- Tests the model's text generation capability
- Provides performance baseline for comparison

## Step 9: Apply Quantization

In [15]:
# Apply 8-bit integer quantization to weights only
quantize(model, weights=qint8, activations=None)

In [16]:
# Freeze the model to finalize quantization
freeze(model)

**Parameters:**
- `weights=qint8`: Quantizes weights to 8-bit integers
- `activations=None`: Keeps activations in original precision
- `freeze()`: Finalizes the quantization process

**What it does:**
- Converts model weights from float32 to int8
- Reduces model size by approximately 75%
- Maintains model functionality while using less memory

## Step 10: Inspect Quantized Model

In [17]:
'''
Layer-by-layer schema of the quantized model.
Note how all the original linear layers have turn
into quantized linear layers (QLinear).
'''
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): QLinear(in_features=2048, out_features=2048, bias=False)
          (k_proj): QLinear(in_features=2048, out_features=512, bias=False)
          (v_proj): QLinear(in_features=2048, out_features=512, bias=False)
          (o_proj): QLinear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): QLinear(in_features=2048, out_features=8192, bias=False)
          (up_proj): QLinear(in_features=2048, out_features=8192, bias=False)
          (down_proj): QLinear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)


**What it does:**
- Shows the quantized model structure
- Displays how weight tensors have changed after quantization
- Allows comparison with original model structure

In [18]:
print(model.model.layers[0].self_attn.q_proj.weight)

<class 'optimum.quanto.tensor.weights.qbytes.WeightQBytesTensor'>(tensor([[-44,  16,  61,  ..., -22, -29,  50],
        [ 14,  70,  65,  ..., -39, -18,  13],
        [ 16,  14,  30,  ..., -34, -34, -24],
        ...,
        [ 17,  20,  40,  ..., -40, -15, -16],
        [ 32, -35,  50,  ..., -17, -41, -21],
        [-14, -30,  -7,  ...,  30,   5,  -2]], device='cuda:0',
       dtype=torch.int8), scale=tensor([[0.0004],
        [0.0009],
        [0.0011],
        ...,
        [0.0018],
        [0.0007],
        [0.0018]], device='cuda:0', dtype=torch.bfloat16), dtype=torch.bfloat16)


## Step 11: Analyze Quantized Model Size

In [19]:
# Calculate size reduction achieved
module_sizes = compute_module_sizes(model)
print(module_sizes)
print(f"The model size is {module_sizes[''] * 1e-9} GB")

defaultdict(<class 'int'>, {'': 1762229444, 'model': 1499304640, 'model.embed_tokens': 525336576, 'model.embed_tokens.weight': 525336576, 'model.layers': 973963712, 'model.layers.0': 60872732, 'model.layers.0.self_attn': 10496016, 'model.layers.0.self_attn.q_proj': 4198404, 'model.layers.0.self_attn.q_proj.weight': 4198400, 'model.layers.0.self_attn.q_proj.weight._data': 4194304, 'model.layers.0.self_attn.q_proj.weight._scale': 4096, 'model.layers.0.self_attn.k_proj': 1049604, 'model.layers.0.self_attn.k_proj.weight': 1049600, 'model.layers.0.self_attn.k_proj.weight._data': 1048576, 'model.layers.0.self_attn.k_proj.weight._scale': 1024, 'model.layers.0.self_attn.v_proj': 1049604, 'model.layers.0.self_attn.v_proj.weight': 1049600, 'model.layers.0.self_attn.v_proj.weight._data': 1048576, 'model.layers.0.self_attn.v_proj.weight._scale': 1024, 'model.layers.0.self_attn.o_proj': 4198404, 'model.layers.0.self_attn.o_proj.weight': 4198400, 'model.layers.0.self_attn.o_proj.weight._data': 41943

**Expected result:** Model size should be approximately 25% of original size due to 8-bit quantization.

## Step 12: Test Quantized Model Performance

In [20]:
print("Before generate:")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e6:.2f} MB")

# Test generation with quantized model
text = "What is Machine Learning?"
inputs = tokenizer(text, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

print("After generate:")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e6:.2f} MB")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Before generate:
Allocated: 1770.87 MB
What is Machine Learning? And How Does it Work?
Machine learning is a subset of artificial intelligence that enables computers to learn from data without being explicitly programmed. It involves training algorithms to make predictions, classify objects, or complete tasks based on patterns and relationships learned from data.

Here
After generate:
Allocated: 1770.87 MB


**What it does:**
- Tests quantized model performance
- Compares memory usage with original model
- Verifies that quantization doesn't significantly affect output quality

## Step 13: Integration in transformers - Alternative Quantization Method

`Quanto` is seamlessly integrated in the `Hugging Face transformers` library. You can quantize any model by passing a `QuantoConfig` to `from_pretrained`!

Currently, you need to use the latest version of `accelerate` to make sure the integration is fully compatible.

In [21]:
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig

# Create quantization configuration
quantization_config = QuantoConfig(weights="int8")

# Load pre-quantized model
quantized_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

**Alternative approach:**
- `QuantoConfig`: Configuration object for quantization settings
- `device_map="auto"`: Automatically distributes model across available devices
- Quantizes model during loading rather than post-loading

## Step 14: Test Alternative Quantization

**What it does:**
- Tests the model quantized using the alternative method
- Compares results with the first quantization approach
- Demonstrates different quantization workflows

In [24]:
text = "What is Machine Learning?"
device = "cuda"

inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = quantized_model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


What is Machine Learning? Machine learning is a subset of artificial intelligence that enables computers to learn from data and improve their performance on a task without being explicitly programmed. In other words, it's a way for computers to automatically improve their abilities by learning from data.

Machine learning is


## Quantization Results Analysis

### Model Size Comparison
- **Original Model Size**: 4.94 GB
- **Quantized Model Size**: 2.29 GB
- **Size Reduction**: 53.7% (2.65 GB saved)
- **Compression Ratio**: 2.16:1

### Memory Usage Comparison

#### Original Model Performance:
```
Before generate: 4943.26 MB allocated
After generate: 4951.78 MB allocated
Memory increase during generation: 8.52 MB
```

#### Quantized Model Performance:
```
Before generate: 2297.35 MB allocated
After generate: 2297.35 MB allocated
Memory increase during generation: 0 MB
```

### Performance Analysis

**Memory Efficiency Gains:**
- **Base Memory Reduction**: 53.5% less GPU memory required (4943.26 MB → 2297.35 MB)
- **Generation Stability**: Quantized model shows no additional memory allocation during generation
- **Total Memory Saved**: 2.65 GB of GPU memory freed up

**Output Quality Comparison:**
- **Original Output**: "Machine learning is a subset of artificial intelligence that involves training algorithms to learn from data, without being explicitly programmed..."
- **Quantized Output**: "Machine learning is a subfield of artificial intelligence that involves training algorithms to make predictions or decisions based on data..."
- **Quality Assessment**: Both outputs are coherent and accurate, with slightly different phrasing but equivalent meaning

## Key Benefits of Quantization

1. **Significant Memory Savings**: Achieved 53.7% reduction in model size (4.94 GB → 2.29 GB)
2. **GPU Memory Efficiency**: Reduced GPU memory usage by over 2.6 GB
3. **Stable Memory Usage**: No additional memory allocation during inference
4. **Maintained Output Quality**: Minimal impact on response coherence and accuracy
5. **Cost Effectiveness**: Enables deployment on smaller GPU configurations
6. **Faster Loading**: Smaller model files load faster from storage

## Real-World Impact

**Before Quantization:**
- Requires ~5GB GPU memory minimum
- Higher infrastructure costs
- Longer model loading times
- Memory spikes during generation

**After Quantization:**
- Requires ~2.3GB GPU memory minimum
- Reduced infrastructure costs (can run on smaller GPUs)
- Faster model loading
- More stable memory usage patterns

## Important Notes

- **Model Selection**: Use Llama if you have access; otherwise, Pythia is a good alternative
- **Authentication**: HuggingFace CLI login is only needed for gated models
- **GPU Memory**: Monitor memory usage to understand quantization benefits
- **Quality Trade-off**: Results show maintained quality with different but equivalent responses
- **Memory Stability**: Quantized models often show more stable memory usage patterns

## Troubleshooting

- If you encounter memory issues, try using a smaller model
- Ensure CUDA is available for GPU acceleration
- For authentication issues, verify your HuggingFace token permissions
- If quantization fails, check if your GPU has sufficient memory for the process